### 1. Install PhysioNet Downloader, MNE-Python, and TensorFlow.js Converter
We will need `wfdb` (to download/interact with PhysioNet), `mne` (to read EDF files which store EEG/EOG signals), and `tensorflowjs` to convert the trained model.

In [9]:
import os
# Force Protobuf to use pure-Python implementation globally before any import happens
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

!pip install -q wfdb mne tensorflow tensorflowjs setuptools

### 2. Download a sample of Sleep-EDFx (v1.0.0)
We will download a couple of sample recordings to demonstrate how to load and prepare the data.

In [10]:
import os
import urllib.request

# Create a directory for the raw data
os.makedirs('sleep_edfx_data', exist_ok=True)

# Direct HTTPS download links from PhysioNet sleep-edfx v1.0.0
base_url = "https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/"
files = ['SC4001E0-PSG.edf', 'SC4001EC-Hypnogram.edf']

for file_name in files:
    target_path = os.path.join('sleep_edfx_data', file_name)
    if not os.path.exists(target_path):
        print(f"Downloading {file_name}...")
        urllib.request.urlretrieve(base_url + file_name, target_path)
        print(f"Finished downloading {file_name}")
    else:
        print(f"{file_name} already exists.")

SC4001E0-PSG.edf already exists.
SC4001EC-Hypnogram.edf already exists.


### 3. Parse the EDF signals and Sleep Stages (Labels)
We use the `mne` library to read raw EEG signals and extract labeled epochs corresponding to different sleep stages (W, N1, N2, N3, REM).

In [11]:
import mne
import numpy as np

# Path to the downloaded files
psg_file = 'sleep_edfx_data/SC4001E0-PSG.edf'
hyp_file = 'sleep_edfx_data/SC4001EC-Hypnogram.edf'

# Load raw PSG (EEG, EOG, EMG)
raw = mne.io.read_raw_edf(psg_file, preload=True)

# Load Annotations (Hypnogram sleep stages)
annotations = mne.read_annotations(hyp_file)
raw.set_annotations(annotations, emit_warning=False)

# Map sleep stages to integers
annotation_desc_2_event_id = {
    'Sleep stage W': 1,
    'Sleep stage 1': 2,
    'Sleep stage 2': 3,
    'Sleep stage 3': 4,
    'Sleep stage 4': 4, # Often combined with stage 3 as N3
    'Sleep stage R': 5
}

events, _ = mne.events_from_annotations(raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0)

# Extract 30-second epochs (typically used in sleep staging)
tmin, tmax = 0., 30. - 1. / raw.info['sfreq']
epochs = mne.Epochs(raw=raw, events=events, event_id=None,
                    tmin=tmin, tmax=tmax, baseline=None, preload=True)

# Get the feature arrays (signals) and targets (stages)
X = epochs.get_data()  # Shape: (n_epochs, n_channels, n_times)
y = epochs.events[:, 2] - 1  # Standardize labels to 0-indexed

print(f"Signals shape: {X.shape} (epochs, channels, samples)")
print(f"Labels shape: {y.shape}")

Extracting EDF parameters from sleep_edfx_data/SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...


/tmp/ipykernel_5079/2551025205.py:9: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True)
/tmp/ipykernel_5079/2551025205.py:9: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True)
/tmp/ipykernel_5079/2551025205.py:9: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True)


Used Annotations descriptions: ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage 4', 'Sleep stage R', 'Sleep stage W']
Not setting metadata
2650 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2650 events and 3000 original time points ...
0 bad epochs dropped
Signals shape: (2650, 7, 3000) (epochs, channels, samples)
Labels shape: (2650,)


### 4. Build and Export a Keras Model to TensorFlow.js
Here is a simple CNN architecture to classify the sleep stage sequence, followed by exporting it for web-browser usage.

In [12]:
import json, os
import numpy as np
from tensorflow import keras

_KI = {"class_name": "VarianceScaling",
       "config": {"scale": 1, "mode": "fan_avg", "distribution": "normal", "seed": None}}
_BI = {"class_name": "Zeros", "config": {}}
_NONE = dict(kernel_regularizer=None, bias_regularizer=None, activity_regularizer=None,
             kernel_constraint=None, bias_constraint=None)

def export_tfjs(model, out_dir, meta=None):
    """Write a TF.js layers-model (model.json + weights.bin) from a Keras 3 Sequential.
    Supports Conv1D, MaxPooling1D, GlobalAveragePooling1D, Flatten, Dense, Dropout.
    No tensorflowjs package needed."""
    os.makedirs(out_dir, exist_ok=True)
    in_shape = [None] + [int(d) for d in model.input_shape[1:]]
    layers, specs, chunks, counts, first = [], [], [], {}, True

    def nm(kind):
        counts[kind] = counts.get(kind, 0) + 1
        return f"{kind.lower()}_{kind}{counts[kind]}"

    def add_weights(name, layer):
        kernel, bias = [w.astype("<f4") for w in layer.get_weights()]
        specs.extend([{"name": f"{name}/kernel", "shape": list(kernel.shape), "dtype": "float32"},
                      {"name": f"{name}/bias", "shape": list(bias.shape), "dtype": "float32"}])
        chunks.extend([kernel.tobytes(), bias.tobytes()])

    for layer in model.layers:
        if isinstance(layer, keras.layers.InputLayer):
            continue
        cls = type(layer).__name__
        extra = {}
        if first:
            extra = {"batch_input_shape": in_shape, "dtype": "float32"}; first = False
        if cls == "Conv1D":
            name = nm("Conv1D")
            cfg = {"filters": int(layer.filters), "kernel_size": [int(layer.kernel_size[0])],
                   "strides": [int(layer.strides[0])], "padding": layer.padding,
                   "data_format": "channels_last", "dilation_rate": [int(layer.dilation_rate[0])],
                   "activation": layer.activation.__name__, "use_bias": True,
                   "kernel_initializer": _KI, "bias_initializer": _BI, **_NONE,
                   "name": name, "trainable": True, **extra}
            add_weights(name, layer)
        elif cls == "Dense":
            name = nm("Dense")
            cfg = {"units": int(layer.units), "activation": layer.activation.__name__, "use_bias": True,
                   "kernel_initializer": _KI, "bias_initializer": _BI, **_NONE,
                   "name": name, "trainable": True, **extra}
            add_weights(name, layer)
        elif cls == "MaxPooling1D":
            cfg = {"pool_size": [int(layer.pool_size[0])], "padding": layer.padding,
                   "strides": [int(layer.strides[0])], "name": nm("MaxPooling1D"),
                   "trainable": True, **extra}
        elif cls == "GlobalAveragePooling1D":
            cfg = {"name": nm("GlobalAveragePooling1D"), "trainable": True, **extra}
        elif cls == "Flatten":
            cfg = {"name": nm("Flatten"), "trainable": True, **extra}
        elif cls == "Dropout":
            cfg = {"rate": float(layer.rate), "noise_shape": None, "seed": None,
                   "name": nm("Dropout"), "trainable": True, **extra}
        else:
            raise ValueError(f"Layer type {cls} not supported by this exporter")
        layers.append({"class_name": cls, "config": cfg})

    model_json = {
        "modelTopology": {"class_name": "Sequential",
                          "config": {"name": "exported_model", "layers": layers},
                          "keras_version": "2.15.0", "backend": "tensorflow"},
        "format": "layers-model", "generatedBy": "manual exporter", "convertedBy": None,
        "weightsManifest": [{"paths": ["weights.bin"], "weights": specs}],
    }
    with open(f"{out_dir}/model.json", "w") as f: json.dump(model_json, f)
    with open(f"{out_dir}/weights.bin", "wb") as f: f.write(b"".join(chunks))
    if meta is not None:
        with open(f"{out_dir}/meta.json", "w") as f: json.dump(meta, f, indent=2)
    print("Exported:", os.listdir(out_dir), f"| weights {os.path.getsize(out_dir + '/weights.bin') / 1024:.0f} KB")